In [ ]:
# ============================================
# DEEPFAKE DETECTION - KAGGLE PREPROCESSING
# ============================================
# Preprocesses raw video datasets into face crops.
# Handles combined upload (DFDC_Dataset_02 + DFD_01 in one Kaggle dataset).
# Outputs TWO preprocessed datasets ready for multi-dataset training.
#
# USAGE:
# 1. Add raw video dataset "DFDC_Dataset_02+DFD_01" as Kaggle input
# 2. Set NUM_FRAMES below (16 or 32)
# 3. Enable GPU accelerator
# 4. Run the notebook
# 5. Download preprocessed_DFDC02_{T}.tar.gz and preprocessed_DFD01_{T}.tar.gz
#
# Then upload both as separate Kaggle datasets for training.

# =============================================
# CONFIGURATION
# =============================================
NUM_FRAMES = 32       # Number of frames per video (T=16 or T=32)
OUTPUT_SIZE = 224     # Face crop size
MIN_FACE_CONF = 0.90  # MTCNN confidence threshold
MIN_DETECTION_RATIO = 0.55

import subprocess, sys, os, time

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch',
                'opencv-python-headless', 'tqdm'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Find raw video datasets
print('\n=== Searching for raw video datasets ===')
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}

def find_raw_datasets():
    """Find directories containing raw videos with real/fake structure."""
    datasets = []
    for root, dirs, files in os.walk('/kaggle/input'):
        # Look for directories that have real/fake (or original/manipulated) subdirs with videos
        label_dirs = [d for d in dirs if d.lower() in ('real', 'fake', 'original', 'manipulated', 'actors', 'altered', 'deepfake', 'df')]
        if not label_dirs:
            continue
        has_videos = False
        video_count = 0
        for ld in label_dirs:
            subdir = os.path.join(root, ld)
            if os.path.isdir(subdir):
                for sr, sd, sf in os.walk(subdir):
                    vc = sum(1 for f in sf if os.path.splitext(f)[1].lower() in VIDEO_EXTS)
                    video_count += vc
                    if vc > 0:
                        has_videos = True
        if has_videos:
            datasets.append((root, video_count))
            print(f'  Found: {root} ({video_count} videos)')
    return datasets

raw_datasets = find_raw_datasets()

if len(raw_datasets) == 0:
    print('ERROR: No raw video datasets found!')
    print('Listing /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    raise RuntimeError('No raw video datasets found!')

# Identify DFDC02 and DFD01 by path keywords
dfdc02_path = None
dfd01_path = None
for path, count in raw_datasets:
    path_lower = path.lower()
    if 'dfdc' in path_lower or 'dataset_02' in path_lower:
        dfdc02_path = path
        print(f'\nIdentified DFDC02: {path} ({count} videos)')
    elif 'dfd' in path_lower and 'dfdc' not in path_lower:
        dfd01_path = path
        print(f'Identified DFD01:  {path} ({count} videos)')

# Fallback: if only one found or names unclear, assign by order
if dfdc02_path is None and dfd01_path is None and len(raw_datasets) >= 2:
    dfdc02_path = raw_datasets[0][0]
    dfd01_path = raw_datasets[1][0]
    print(f'\nFallback assignment:')
    print(f'  DFDC02: {dfdc02_path}')
    print(f'  DFD01:  {dfd01_path}')

# Build list of datasets to process
to_process = []
if dfdc02_path:
    to_process.append(('DFDC02', dfdc02_path))
if dfd01_path:
    to_process.append(('DFD01', dfd01_path))

if not to_process:
    # Single dataset fallback
    to_process.append(('DATASET', raw_datasets[0][0]))

print(f'\nWill preprocess {len(to_process)} dataset(s) with T={NUM_FRAMES}:')
for name, path in to_process:
    print(f'  {name}: {path}')

# Step 4: Run preprocessing for each dataset
for ds_name, ds_path in to_process:
    output_name = f'preprocessed_{ds_name}_{NUM_FRAMES}'
    output_path = f'/kaggle/working/{output_name}'
    
    print(f'\n{"="*60}')
    print(f'PREPROCESSING: {ds_name}')
    print(f'  Input:  {ds_path}')
    print(f'  Output: {output_path}')
    print(f'  Frames: {NUM_FRAMES}')
    print(f'{"="*60}\n')
    
    t0 = time.time()
    
    # Detect GPU
    gpu_check = subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.is_available())'],
                               capture_output=True, text=True)
    device = 'cuda' if gpu_check.stdout.strip() == 'True' else 'cpu'
    
    cmd = [
        sys.executable, '-u', '/kaggle/working/project/preprocess_videos.py',
        ds_path, output_path,
        '--max-frames', str(NUM_FRAMES),
        '--output-size', str(OUTPUT_SIZE),
        '--min-face-confidence', str(MIN_FACE_CONF),
        '--min-detection-ratio', str(MIN_DETECTION_RATIO),
        '--min-saved-faces', str(NUM_FRAMES),
        '--device', device,
    ]
    
    print(f'Command: {" ".join(cmd)}')
    proc = subprocess.Popen(cmd, cwd='/kaggle/working/project',
                            env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            bufsize=1, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    
    elapsed = (time.time() - t0) / 60
    print(f'\n{ds_name} preprocessing done in {elapsed:.1f} min (exit code: {proc.returncode})')
    
    # Show summary
    import json
    summary_path = os.path.join(output_path, 'summary.json')
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            summary = json.load(f)
        print(f'\n--- {ds_name} SUMMARY ---')
        print(f'Total videos: {summary["total_videos"]}')
        print(f'Saved:   {summary["saved_videos"]} (real={summary["real_saved"]}, fake={summary["fake_saved"]})')
        print(f'Dropped: {summary["dropped_videos"]}')
        print(f'Errors:  {summary["error_videos"]}')
    
    # Package for download
    print(f'\nPackaging {output_name}...')
    os.system(f'tar czf /kaggle/working/{output_name}.tar.gz -C /kaggle/working {output_name}/')
    tar_size = os.path.getsize(f'/kaggle/working/{output_name}.tar.gz') / (1024**2)
    print(f'{output_name}.tar.gz: {tar_size:.1f} MB')

print(f'\n{"="*60}')
print('ALL PREPROCESSING COMPLETE')
print(f'{"="*60}')
print('\nOutput files ready for download:')
for ds_name, _ in to_process:
    fname = f'preprocessed_{ds_name}_{NUM_FRAMES}.tar.gz'
    fpath = f'/kaggle/working/{fname}'
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / (1024**2)
        print(f'  {fname} ({size:.1f} MB)')
print('\nUpload these as separate Kaggle datasets, then run the training notebook.')